[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/41.%20The%20Terminal%20Concession%20Bidding%20Problem/P41-Tier-2_executed.ipynb)

---


# 41. The Terminal Concession Bidding Problem

## Tier 2: Constructive Multi-Criteria Evaluation Heuristic

### Goal and workflow

This notebook implements a constructive, three-stage evaluation procedure for terminal concession bids:

1. **Qualification filtering:** reject any bidder that fails at least one minimum criterion threshold.
2. **Weighted scoring:** calculate a weighted sum for each remaining bidder.
3. **Tie-breaking:** when primary scores are equal within a stated numerical tolerance, rank those bidders by a secondary score.

The procedure returns a ranked list, the winning bidder when at least one bidder qualifies, qualification counts, and the measured evaluation time. The example also visualizes the results and examines how the procedure scales as the number of bidders increases.

### Model assumptions and scope

- Every criterion is scored on a comparable points scale, and a higher score is better.
- A bidder qualifies only if it supplies a score for every criterion and meets every corresponding threshold. Meeting a threshold exactly is sufficient.
- The primary score is the weighted sum of criterion scores. The criterion weights are nonnegative and sum to one in the example.
- The secondary score combines financial capacity and market share using fixed coefficients of 0.6 and 0.4, respectively. Those values are used only to order bidders whose primary scores are tied within the numerical tolerance; they are not added to the primary score.
- Other bidder attributes, such as employment commitment and automation investment, are stored in the example data but do not affect qualification or ranking.
- This is a scoring heuristic, not a complete concession-design or integer-programming model. It does not model payment decisions, interactions between bids, uncertainty, or other constraints not explicitly defined here. The notebook does not establish that the result is optimal for a separate Tier 1 model.
- Execution-time measurements depend on the computer and software environment. They are observations, not performance guarantees.

### Mathematical model

Let \(B\) be the set of bidders and \(C\) the set of evaluation criteria. Bidder \(b\)'s score on criterion \(c\) is \(s_{bc}\), the criterion weight is \(w_c\), and the minimum threshold is \(q_c\). A bidder qualifies if and only if

$$
Q = \left\{b \in B : s_{bc} \text{ is present and } s_{bc} \geq q_c \text{ for every } c \in C\right\}.
$$

For each qualified bidder, the primary weighted score is

$$
P_b = \sum_{c \in C} w_c s_{bc}.
$$

The secondary score used to break primary-score ties is

$$
T_b = 0.6\left(\frac{F_b}{100}\right) + 0.4\left(\frac{M_b}{100}\right),
$$

where \(F_b\) is financial capacity and \(M_b\) is market share. The implementation orders bidders by decreasing \(P_b\), then uses decreasing \(T_b\) within groups whose primary scores differ from the group's first score by less than \(10^{-3}\). Thus, the secondary score does not change the primary score reported in the rankings. If secondary scores also tie, Python's stable sorting retains the bidders' input order.

If no bidder qualifies, the ranking is empty and there is no winner. Otherwise, the first bidder in the final ranking is the winner. The procedure is constructive: it filters the input list, calculates scores for the survivors, sorts them, and returns the resulting ranking.

### Computational cost

For \(n=|B|\) bidders and \(m=|C|\) criteria, checking qualification and computing scores take \(O(nm)\) time. Sorting at most \(n\) qualified bidders takes \(O(n\log n)\) time, so the total worst-case time is \(O(nm+n\log n)\). The algorithm stores scores and rankings for bidders, requiring \(O(n)\) additional space beyond the input data; the input score data itself contains up to \(O(nm)\) values. These bounds describe growth with input size, not a promised wall-clock duration.

### What to examine

- Which bidders pass every threshold, and why others fail.
- How each criterion contributes to the primary weighted score.
- Whether the secondary score is used to order a primary-score tie.
- How the measured evaluation time changes across the generated scalability instances.
- Why agreement with the maximum primary score in this same scoring model is a consistency check, not evidence of optimality for a different optimization model.

In [ ]:
from dataclasses import dataclass
import math
import time
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class BidderHeuristic:
    """Bidder attributes used by the evaluation procedure and example."""

    name: str
    scores: Dict[str, float]
    financial_capacity: float
    employment_commitment: int
    automation_investment: float
    market_share: float


@dataclass
class EvaluationCriterion:
    """A primary scoring criterion and its qualification threshold."""

    name: str
    weight: float
    min_threshold: float


@dataclass
class HeuristicResult:
    """Evaluation outcome, including an empty outcome when nobody qualifies."""

    winner: Optional[str]
    final_score: float
    rankings: List[Tuple[str, float]]
    qualified_count: int
    disqualified_count: int
    execution_time: float
    tie_breaking_used: bool

## 1. Implement the constructive evaluation procedure

The first stage checks every bidder against every criterion. Missing criterion scores fail qualification because the model requires a score for each criterion. The second stage calculates \(P_b\) only for qualified bidders. The final stage sorts primary scores and applies the secondary score within numerical tie groups.

The \(10^{-3}\) tolerance is an absolute tolerance: two primary scores are treated as tied when their absolute difference from the first score in the group is less than \(10^{-3}\). This makes the comparison explicit for floating-point calculations. The tolerance affects tie handling, not the primary score formula.

In [ ]:
class TerminalConcessionHeuristic:
    """Constructive evaluator for terminal concession bids."""

    TIE_TOLERANCE = 1e-3

    def __init__(self, criteria: List[EvaluationCriterion]):
        self.criteria = criteria
        self.min_thresholds = {criterion.name: criterion.min_threshold for criterion in criteria}

    def qualification_filtering(
        self, bidders: List[BidderHeuristic]
    ) -> Tuple[List[BidderHeuristic], List[str]]:
        """Keep bidders meeting every threshold and report each failure."""
        qualified_bidders = []
        disqualified_reasons = []

        for bidder in bidders:
            failed_criteria = []

            for criterion_name, min_score in self.min_thresholds.items():
                if criterion_name not in bidder.scores:
                    failed_criteria.append(f"{criterion_name} (missing)")
                elif bidder.scores[criterion_name] < min_score:
                    failed_criteria.append(
                        f"{criterion_name} ({bidder.scores[criterion_name]} < {min_score})"
                    )

            if failed_criteria:
                disqualified_reasons.append(
                    f"{bidder.name}: {'; '.join(failed_criteria)}"
                )
            else:
                qualified_bidders.append(bidder)

        return qualified_bidders, disqualified_reasons

    def calculate_weighted_score(self, bidder: BidderHeuristic) -> float:
        """Calculate the primary weighted sum for a qualified bidder."""
        return sum(
            bidder.scores[criterion.name] * criterion.weight
            for criterion in self.criteria
        )

    def calculate_tie_breaker_score(self, bidder: BidderHeuristic) -> float:
        """Calculate the secondary score used only for primary-score ties."""
        financial_score = (bidder.financial_capacity / 100.0) * 0.6
        market_score = (bidder.market_share / 100.0) * 0.4
        return financial_score + market_score

    def weighted_scoring(
        self, qualified_bidders: List[BidderHeuristic]
    ) -> List[Tuple[str, float, float]]:
        """Score qualified bidders and order them by primary score."""
        bidder_scores = []

        for bidder in qualified_bidders:
            primary_score = self.calculate_weighted_score(bidder)
            tie_breaker_score = self.calculate_tie_breaker_score(bidder)
            bidder_scores.append((bidder.name, primary_score, tie_breaker_score))

        bidder_scores.sort(key=lambda item: (item[1], item[2]), reverse=True)
        return bidder_scores

    def handle_tie_breaking(
        self, bidder_scores: List[Tuple[str, float, float]]
    ) -> Tuple[List[Tuple[str, float]], bool]:
        """Apply the secondary score within primary-score tolerance groups."""
        final_rankings = []
        tie_breaking_used = False
        i = 0

        while i < len(bidder_scores):
            current_bidder = bidder_scores[i]
            tied_bidders = [current_bidder]
            j = i + 1

            while (
                j < len(bidder_scores)
                and abs(bidder_scores[j][1] - current_bidder[1]) < self.TIE_TOLERANCE
            ):
                tied_bidders.append(bidder_scores[j])
                j += 1

            if len(tied_bidders) > 1:
                tie_breaking_used = True
                tied_bidders.sort(key=lambda item: item[2], reverse=True)

            final_rankings.extend(
                (bidder_name, primary_score)
                for bidder_name, primary_score, _ in tied_bidders
            )
            i = j

        return final_rankings, tie_breaking_used

    def evaluate_concession_bidding(
        self, bidders: List[BidderHeuristic]
    ) -> HeuristicResult:
        """Run qualification, weighted scoring, and tie-breaking in sequence."""
        start_time = time.perf_counter()

        qualified_bidders, disqualified_reasons = self.qualification_filtering(bidders)
        bidder_scores = self.weighted_scoring(qualified_bidders)
        final_rankings, tie_breaking_used = self.handle_tie_breaking(bidder_scores)

        winner = final_rankings[0][0] if final_rankings else None
        final_score = final_rankings[0][1] if final_rankings else 0.0
        execution_time = time.perf_counter() - start_time

        return HeuristicResult(
            winner=winner,
            final_score=final_score,
            rankings=final_rankings,
            qualified_count=len(qualified_bidders),
            disqualified_count=len(disqualified_reasons),
            execution_time=execution_time,
            tie_breaking_used=tie_breaking_used,
        )

## 2. Create the example instance

The example has four criteria, each with a threshold of 70 points. Their weights sum to one, so a weighted score is on the same points scale as the criterion scores. A bidder must meet all four thresholds to enter the ranking.

In [ ]:
def create_heuristic_example():
    """Create the four-bidder terminal concession example."""
    criteria = [
        EvaluationCriterion("Financial Capacity", 0.4, 70),
        EvaluationCriterion("Technical Experience", 0.3, 70),
        EvaluationCriterion("Revenue Commitment", 0.2, 70),
        EvaluationCriterion("Performance Guarantees", 0.1, 70),
    ]

    bidders = [
        BidderHeuristic(
            name="Mediterranean_TC",
            scores={
                "Financial Capacity": 85,
                "Technical Experience": 90,
                "Revenue Commitment": 80,
                "Performance Guarantees": 95,
            },
            financial_capacity=85,
            employment_commitment=280,
            automation_investment=45,
            market_share=12.5,
        ),
        BidderHeuristic(
            name="Global_Port_Holdings",
            scores={
                "Financial Capacity": 95,
                "Technical Experience": 85,
                "Revenue Commitment": 90,
                "Performance Guarantees": 85,
            },
            financial_capacity=95,
            employment_commitment=320,
            automation_investment=65,
            market_share=18.2,
        ),
        BidderHeuristic(
            name="European_Maritime",
            scores={
                "Financial Capacity": 80,
                "Technical Experience": 95,
                "Revenue Commitment": 85,
                "Performance Guarantees": 90,
            },
            financial_capacity=80,
            employment_commitment=295,
            automation_investment=55,
            market_share=15.8,
        ),
        BidderHeuristic(
            name="Atlantic_Container",
            scores={
                "Financial Capacity": 65,
                "Technical Experience": 88,
                "Revenue Commitment": 75,
                "Performance Guarantees": 82,
            },
            financial_capacity=65,
            employment_commitment=250,
            automation_investment=35,
            market_share=8.3,
        ),
    ]

    return criteria, bidders


criteria, bidders = create_heuristic_example()
heuristic = TerminalConcessionHeuristic(criteria)

print("Created the example evaluation with:")
print(f"- {len(bidders)} bidders")
print(f"- {len(criteria)} evaluation criteria")
print(
    "- Qualification thresholds: "
    f"{[f'{criterion.name}: {criterion.min_threshold}' for criterion in criteria]}"
)

## 3. Evaluate and inspect the ranking

The following function displays the input profiles and the outcome of the three stages. Its reported execution time is measured when the cell runs; it will vary across runs and computing environments.

In [ ]:
def run_heuristic_evaluation(
    heuristic: TerminalConcessionHeuristic,
    bidders: List[BidderHeuristic],
) -> HeuristicResult:
    """Run the evaluation and print profiles, counts, and rankings."""
    print("=== Terminal Concession Bidding Evaluation ===")
    print(f"\nInitial bidders: {len(bidders)}")
    print("\nInitial bidder profiles:")

    for bidder in bidders:
        scores_str = ", ".join(
            f"{criterion}: {score}" for criterion, score in bidder.scores.items()
        )
        print(f"- {bidder.name}: {scores_str}")

    result = heuristic.evaluate_concession_bidding(bidders)
    print(f"\nQualified bidders: {result.qualified_count} out of {len(bidders)}")
    print(f"Disqualified bidders: {result.disqualified_count}")

    if result.winner is None:
        print("\nWinner: None (no bidder qualified)")
    else:
        print(f"\nWinner: {result.winner}")
        print(f"Winning primary score: {result.final_score:.2f}")

    print(f"Tie-breaking used: {'Yes' if result.tie_breaking_used else 'No'}")
    print("\nRankings:")
    if result.rankings:
        for rank, (name, score) in enumerate(result.rankings, start=1):
            print(f"{rank}. {name}: {score:.2f}")
    else:
        print("No qualified bidders to rank.")

    print("\nMeasured evaluation time:")
    print(f"- {result.execution_time:.6f} seconds")
    return result


result = run_heuristic_evaluation(heuristic, bidders)

assert result.qualified_count + result.disqualified_count == len(bidders)
assert (result.winner is None) == (result.qualified_count == 0)
if result.winner is not None:
    assert result.rankings[0][0] == result.winner
    assert math.isclose(result.rankings[0][1], result.final_score)

## 4. Visualize qualification, scores, and score contributions

The first panel distinguishes qualification status. The second shows primary weighted scores for qualified bidders; the winning bidder is highlighted. The third panel shows a theoretical operation-count proxy based on the derived complexity bound. It is not a wall-clock benchmark. The final panel displays how each criterion contributes to the winner's primary score.

In [ ]:
def visualize_heuristic_results(
    heuristic: TerminalConcessionHeuristic,
    bidders: List[BidderHeuristic],
    result: HeuristicResult,
):
    """Plot qualification, weighted scores, complexity, and winner contributions."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "Terminal Concession Bidding: Heuristic Evaluation",
        fontsize=16,
        fontweight="bold",
    )

    qualified_bidders, _ = heuristic.qualification_filtering(bidders)
    qualified_names = {bidder.name for bidder in qualified_bidders}

    ax1 = axes[0, 0]
    statuses = [
        "Qualified" if bidder.name in qualified_names else "Disqualified"
        for bidder in bidders
    ]
    colors = ["#2ECC71" if status == "Qualified" else "#E74C3C" for status in statuses]
    bars = ax1.bar([bidder.name for bidder in bidders], [1] * len(bidders), color=colors)
    ax1.set_xlabel("Bidder")
    ax1.set_ylabel("Qualification status")
    ax1.set_title("Qualification filtering")
    ax1.set_ylim(0, 1.2)
    ax1.set_xticks(range(len(bidders)))
    ax1.set_xticklabels([bidder.name for bidder in bidders], rotation=30, ha="right")
    for bar, status in zip(bars, statuses):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            0.5,
            status,
            ha="center",
            va="center",
            fontweight="bold",
            color="white",
        )

    ax2 = axes[0, 1]
    score_rows = [
        (bidder.name, heuristic.calculate_weighted_score(bidder))
        for bidder in qualified_bidders
    ]
    score_rows.sort(key=lambda row: row[1], reverse=True)
    score_names = [row[0] for row in score_rows]
    score_values = [row[1] for row in score_rows]
    score_colors = [
        "#F39C12" if name == result.winner else "#3498DB" for name in score_names
    ]
    score_bars = ax2.bar(score_names, score_values, color=score_colors)
    for bar, score in zip(score_bars, score_values):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{score:.2f}",
            ha="center",
            va="bottom",
        )
    ax2.set_xlabel("Qualified bidder")
    ax2.set_ylabel("Primary weighted score (points)")
    ax2.set_title("Weighted-score ranking")
    ax2.tick_params(axis="x", rotation=30)
    ax2.grid(axis="y", alpha=0.3)

    ax3 = axes[1, 0]
    bidder_counts = np.array([5, 10, 20, 50, 100, 200])
    criterion_count = len(heuristic.criteria)
    operation_proxy = (
        bidder_counts * criterion_count
        + bidder_counts * np.log2(np.maximum(bidder_counts, 2))
    )
    ax3.plot(
        bidder_counts,
        operation_proxy,
        "o-",
        color="#8E44AD",
        label=r"Proxy for $nm + n\log_2 n$",
    )
    ax3.set_xlabel("Number of bidders, n")
    ax3.set_ylabel("Relative operation-count proxy")
    ax3.set_title("Theoretical growth (not measured time)")
    ax3.set_xscale("log")
    ax3.set_yscale("log")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    if result.winner is not None:
        winner_bidder = next(
            bidder for bidder in qualified_bidders if bidder.name == result.winner
        )
        criterion_names = [criterion.name for criterion in heuristic.criteria]
        contributions = [
            winner_bidder.scores[criterion.name] * criterion.weight
            for criterion in heuristic.criteria
        ]
        contribution_colors = ["#3498DB", "#E74C3C", "#F39C12", "#2ECC71"]
        ax4.bar(
            criterion_names,
            contributions,
            color=contribution_colors[: len(criterion_names)],
        )
        for index, (contribution, criterion) in enumerate(
            zip(contributions, heuristic.criteria)
        ):
            ax4.text(
                index,
                contribution,
                f"{contribution:.2f}",
                ha="center",
                va="bottom",
            )
        ax4.set_xticks(range(len(criterion_names)))
        ax4.set_xticklabels(criterion_names, rotation=35, ha="right")
        ax4.set_title(f"Winner's score contributions: {result.winner}")
    else:
        ax4.set_title("No qualified bidder; no score breakdown")
    ax4.set_xlabel("Evaluation criterion")
    ax4.set_ylabel("Contribution to primary score (points)")
    ax4.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_heuristic_results(heuristic, bidders, result)

## 5. Check consistency with the primary scoring rule

For this model, the highest primary weighted score among qualified bidders must be represented at the top of the ranking, allowing for the stated tie tolerance. This check verifies that the implemented ranking follows its own score rule. It is not a comparison with integer programming, does not test other constraints, and cannot establish optimality for another model.

In [ ]:
def check_primary_score_consistency(
    heuristic: TerminalConcessionHeuristic,
    bidders: List[BidderHeuristic],
    result: HeuristicResult,
) -> Dict[str, Optional[float]]:
    """Check whether the winner's score matches the maximum primary score."""
    qualified_bidders, _ = heuristic.qualification_filtering(bidders)

    if not qualified_bidders:
        print("No qualified bidders; primary-score consistency is not applicable.")
        return {"maximum_primary_score": None, "winner_primary_score": None}

    maximum_primary_score = max(
        heuristic.calculate_weighted_score(bidder) for bidder in qualified_bidders
    )
    winner_primary_score = result.final_score
    consistent = (
        maximum_primary_score - winner_primary_score
        < heuristic.TIE_TOLERANCE
    )

    print("=== Primary-Score Consistency Check ===")
    print(f"Maximum primary score: {maximum_primary_score:.6f}")
    print(f"Winner's primary score: {winner_primary_score:.6f}")
    print(f"Consistent within tie tolerance: {'Yes' if consistent else 'No'}")

    assert consistent, "The ranking does not follow the stated primary-score rule."
    return {
        "maximum_primary_score": maximum_primary_score,
        "winner_primary_score": winner_primary_score,
    }


score_check = check_primary_score_consistency(heuristic, bidders, result)

## 6. Examine scalability

This experiment generates bidder instances of increasing size and applies the same qualification, weighted-scoring, and tie-breaking procedure. A fixed generator seed makes the generated experiment data reproducible. The measured times are specific to the execution environment; the qualification counts depend on the generated scores and thresholds.

In [ ]:
def scalability_analysis():
    """Measure evaluation time on generated instances of increasing size."""
    print("=== Scalability Analysis ===")
    bidder_counts = [5, 10, 20, 50, 100, 200]
    execution_times = []
    memory_estimates = []
    qualified_counts = []
    rng = np.random.default_rng(41)

    test_criteria = [
        EvaluationCriterion("Financial Capacity", 0.4, 70),
        EvaluationCriterion("Technical Experience", 0.3, 70),
        EvaluationCriterion("Revenue Commitment", 0.2, 70),
        EvaluationCriterion("Performance Guarantees", 0.1, 70),
    ]
    test_heuristic = TerminalConcessionHeuristic(test_criteria)

    for bidder_count in bidder_counts:
        test_bidders = []
        for index in range(bidder_count):
            scores = {
                "Financial Capacity": int(rng.integers(60, 100)),
                "Technical Experience": int(rng.integers(60, 100)),
                "Revenue Commitment": int(rng.integers(60, 100)),
                "Performance Guarantees": int(rng.integers(60, 100)),
            }
            test_bidders.append(
                BidderHeuristic(
                    name=f"Bidder_{index + 1}",
                    scores=scores,
                    financial_capacity=scores["Financial Capacity"],
                    employment_commitment=int(rng.integers(200, 400)),
                    automation_investment=float(rng.integers(30, 80)),
                    market_share=float(rng.uniform(5, 25)),
                )
            )

        start_time = time.perf_counter()
        test_result = test_heuristic.evaluate_concession_bidding(test_bidders)
        elapsed = time.perf_counter() - start_time

        execution_times.append(elapsed)
        qualified_counts.append(test_result.qualified_count)
        # This estimates numeric score storage only, not Python object overhead.
        memory_estimates.append(bidder_count * len(test_criteria) * 8)
        print(
            f"- {bidder_count} bidders: {elapsed:.6f} seconds, "
            f"{test_result.qualified_count} qualified"
        )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(bidder_counts, execution_times, "o-", color="#E74C3C")
    ax1.set_xlabel("Number of bidders")
    ax1.set_ylabel("Measured evaluation time (seconds)")
    ax1.set_title("Measured evaluation time")
    ax1.set_xscale("log")
    ax1.set_yscale("log")
    ax1.grid(True, alpha=0.3)

    ax2.plot(bidder_counts, memory_estimates, "s-", color="#2ECC71")
    ax2.set_xlabel("Number of bidders")
    ax2.set_ylabel("Estimated numeric score storage (bytes)")
    ax2.set_title("Score-storage estimate (excludes object overhead)")
    ax2.set_xscale("log")
    ax2.set_yscale("log")
    ax2.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
    return execution_times, memory_estimates, qualified_counts


exec_times, memory_estimates, qualified_counts = scalability_analysis()

## 7. Summarize the evaluation

The summary brings together the model's stages, the example outcome, the consistency check, and the scalability measurements. Interpret timing values as measurements from the current run, not as guarantees or comparisons with a different optimization method.

In [ ]:
def generate_heuristic_summary(
    heuristic: TerminalConcessionHeuristic,
    bidders: List[BidderHeuristic],
    result: HeuristicResult,
    score_check: Dict[str, Optional[float]],
    bidder_counts: List[int],
    exec_times: List[float],
    qualified_counts: List[int],
) -> None:
    """Print a concise summary of the model and observed results."""
    print("\n" + "=" * 76)
    print("TERMINAL CONCESSION BIDDING: CONSTRUCTIVE HEURISTIC")
    print("=" * 76)
    print("\nAlgorithm characteristics:")
    print("- Stages: qualification filtering, weighted scoring, tie-breaking")
    print("- Worst-case time: O(nm + n log n)")
    print("- Additional ranking space: O(n), beyond the input score data")

    print("\nExample evaluation:")
    print(f"- Total bidders: {len(bidders)}")
    print(f"- Qualified bidders: {result.qualified_count}")
    print(f"- Disqualified bidders: {result.disqualified_count}")
    print(f"- Winner: {result.winner if result.winner is not None else 'None'}")
    print(f"- Winning primary score: {result.final_score:.2f}")
    print(f"- Tie-breaking used: {'Yes' if result.tie_breaking_used else 'No'}")
    print(f"- Measured evaluation time: {result.execution_time:.6f} seconds")

    if score_check["maximum_primary_score"] is not None:
        print("\nPrimary-score consistency check:")
        print(f"- Maximum primary score: {score_check['maximum_primary_score']:.2f}")
        print(f"- Winner's primary score: {score_check['winner_primary_score']:.2f}")
        print("- This check concerns the stated scoring rule only.")

    print("\nScalability observations from this run:")
    for count, elapsed, qualified in zip(
        bidder_counts, exec_times, qualified_counts
    ):
        print(
            f"- {count} bidders: {elapsed:.6f} seconds; "
            f"{qualified} qualified"
        )

    print("\nInterpretation:")
    print(
        "- Qualification is a feasibility screen: one failed or missing criterion "
        "excludes a bidder."
    )
    print(
        "- Weighted scoring ranks only the qualified bidders; the secondary score "
        "is used only for near-equal primary scores."
    )
    print(
        "- The procedure's complexity bound describes asymptotic growth and does "
        "not guarantee a particular runtime."
    )
    print("=" * 76)


generate_heuristic_summary(
    heuristic,
    bidders,
    result,
    score_check,
    [5, 10, 20, 50, 100, 200],
    exec_times,
    qualified_counts,
)